# **CHAIN WITH CUSTOM RUNNABLE** (Groq Version)

Sometimes you need a plain Python function in the middle of a chain.
RunnableLambda wraps any Python function so it plugs into the | pipeline.

In [ ]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

if os.environ.get("GROQ_API_KEY"):
    print("Bro API KEY Variable exists")
else:
    raise ValueError("GROQ_API_KEY not found")

llm_groq = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)


In [ ]:
# TASK 1 — First prompt: ask the LLM a factual question
# {input} will be the user's question (e.g. 'What is the capital of France?')
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant"),
    ("human",  "{input}")
])


In [ ]:
# TASK 2 — LLM for the first step
llm_groq = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)


In [ ]:
# TASK 3 — Parse LLM output to plain string
str_parser = StrOutputParser()


In [ ]:
# TASK 4 — Custom Runnable (the key concept of this notebook)
from langchain_core.runnables import RunnableLambda

# Problem: the NEXT prompt template expects a dict input like {"text": "..."},
# but StrOutputParser gives us a plain string.
# Solution: write a tiny function to wrap the string in a dict.
def dictionary_maker(text: str) -> dict:
    # Receives the plain string from StrOutputParser
    # Returns a dict whose key matches the next prompt's placeholder {text}
    return {"text": text}

# RunnableLambda makes any Python function usable inside a | chain
# Without this wrapper, the | operator wouldn't know how to pipe data into it
dictionary_maker_runnable = RunnableLambda(dictionary_maker)


In [ ]:
# TASK 5 — Second prompt: turn the factual answer into a LinkedIn post
# {text} matches the key returned by dictionary_maker above
prompt_post = ChatPromptTemplate.from_messages(
    messages=[
        ("system", "You're a social media post generator."),
        ("human",  "Create a post for the following text for LinkedIn: {text}"),
    ]
)


In [ ]:
# TASK 6 — LLM for the second step (same model, reused)
llm_groq = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)


In [ ]:
# TASK 7 — Parse the LinkedIn post to a plain string
str_parser = StrOutputParser()


In [ ]:
# FULL CHAIN — read left to right:
#  1. prompt_template  → fills {input} with the user question
#  2. llm_groq         → answers the question → AIMessage
#  3. str_parser       → extracts text string from AIMessage
#  4. dictionary_maker_runnable → wraps string into {"text": "..."}  (format bridge)
#  5. prompt_post      → fills {text} with the answer → LinkedIn prompt
#  6. llm_groq         → writes the LinkedIn post → AIMessage
#  7. str_parser       → extracts final plain string
chain = prompt_template | llm_groq | str_parser | dictionary_maker_runnable | prompt_post | llm_groq | str_parser

# Single .invoke() call runs all 7 steps in sequence automatically
chain.invoke("What is the capital of France?")
